In [ ]:
!pip install mediapipe opencv-python joblib scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 9.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing inst

In [ ]:
import shutil
import glob
import os

# Drunk images from train, valid, test
os.makedirs("drunk", exist_ok=True)

for folder in ['train', 'valid', 'test']:
    paths = glob.glob(f"drunk.v1i.tensorflow/{folder}/*.jpg")
    for p in paths:
        shutil.copy(p, "drunk")


In [ ]:
import os

# Create folders if they don't exist
os.makedirs("dataset/drunk", exist_ok=True)
os.makedirs("dataset/sober", exist_ok=True)


In [ ]:
import zipfile
import glob

# Unzip drunk faces
for zip_path in glob.glob("dataset/drunk/*.zip"):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("dataset/drunk/")
        print(f"✅ Extracted: {zip_path}")

# Unzip sober faces
for zip_path in glob.glob("dataset/sober/*.zip"):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("dataset/sober/")
        print(f"✅ Extracted: {zip_path}")

✅ Extracted: dataset/drunk/train.zip
✅ Extracted: dataset/sober/sobertrain.zip


In [ ]:
import os
import glob
import cv2
import numpy as np
import mediapipe as mp
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Mediapipe Setup
mpfm = mp.solutions.face_mesh.FaceMesh(static_image_mode=True)

# --- Feature Functions ---
def get_landmarks(img):
    res = mpfm.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    return res.multi_face_landmarks[0].landmark if res.multi_face_landmarks else None

def eye_aspect_ratio(lm, img):
    ih, iw = img.shape[:2]
    def d(a,b):
        p1, p2 = lm[a], lm[b]
        return np.linalg.norm([(p1.x-p2.x)*iw, (p1.y-p2.y)*ih])
    vert = (d(159,153) + d(158,144)) / 2
    hor = d(33,133)
    return vert / hor

def cheek_redness(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv[:,:,0], 0, 10)
    return mask.sum()/255 / img.size

def mouth_opening(lm, img):
    ih = img.shape[0]
    return abs(lm[13].y - lm[14].y)*ih

def eye_circularity(lm, img):
    ih, iw = img.shape[:2]
    # Approximate eye region landmarks
    left_eye_top = lm[159]
    left_eye_bottom = lm[145]
    left_eye_left = lm[33]
    left_eye_right = lm[133]

    vertical = abs(left_eye_top.y - left_eye_bottom.y) * ih
    horizontal = abs(left_eye_right.x - left_eye_left.x) * iw

    if horizontal == 0:
        return 0  # Avoid division by zero

    circularity = (vertical / horizontal)
    return circularity

def lip_line_curvature(lm, img):
    ih, iw = img.shape[:2]
    left_corner = lm[61]
    right_corner = lm[291]
    center = lm[13]  # Upper lip center

    # Calculate line slope difference
    left_y = left_corner.y * ih
    right_y = right_corner.y * ih
    center_y = center.y * ih

    curvature = abs((left_y + right_y)/2 - center_y)
    return curvature


def extract_features(img):
    lm = get_landmarks(img)
    if not lm:
        return None
    return [eye_aspect_ratio(lm, img), cheek_redness(img), mouth_opening(lm, img), eye_circularity(lm, img), lip_line_curvature(lm, img)]

# === Always Train Model ===
print("🔁 Training model from scratch...")

# --- Load Training Data ---
X, y = [], []
for label_name, label in [('drunk', 1), ('sober', 0)]:
    for path in glob.glob(f"dataset/{label_name}/*.jpg"):
        img = cv2.imread(path)
        if img is not None:
            feat = extract_features(img)
            if feat:
                X.append(feat)
                y.append(label)

if len(X) < 2:
    raise Exception("❌ Not enough data! Add more images to 'dataset/drunk' and 'dataset/sober'.")

X = np.array(X)
y = np.array(y)

# fallback for small datasets
if len(X) < 3:
    print("⚠️ Not enough samples to split — using all for training.")
    X_train, X_test, y_train, y_test = X, X, y, y
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_train, y_train)
acc = accuracy_score(y_test, clf.predict(X_test))
print(f"✅ Model trained. Test Accuracy: {acc:.2f}")
joblib.dump(clf, "drunk_model.pkl")



🔁 Training model from scratch...
✅ Model trained. Test Accuracy: 1.00


['drunk_model.pkl']

In [ ]:
# === Prediction on Custom Images ===
def predict(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"[!] Image not found: {image_path}")
        return
    feat = extract_features(img)
    if not feat:
        print(f"[!] No face detected in {image_path}")
        return
    result = clf.predict([feat])[0]
    print(f"🔍 {image_path} → {'Drunk' if result else 'Sober'}")

# Predict on your custom test images
test_imgs = ["baseline.jpg", "test.jpg"]
for path in test_imgs:
    if os.path.exists(path):
        predict(path)
    else:
        print(f"[!] Skipped. {path} not found.")

🔍 baseline.jpg → Drunk
🔍 test.jpg → Drunk


In [ ]:
import pandas as pd
df = pd.DataFrame(X, columns=["Eye", "Redness", "Mouth", "eye_circ" , "lip_curve" ])
df["Label"] = y
df.to_csv("features.csv", index=False)